# Grafana Expanded Capabilities

**Course Modules:**
1. Introduction to the Grafana Product Ecosystem
2. Enabling Advanced Storage, Tracing, and Rules
3. Advanced Grafana Integrations and Automation

---
## Module 1 — Introduction to the Grafana Product Ecosystem

### Grafana Loki

Loki is Grafana's log aggregation system, designed to be cost-effective and simple to operate.

| Feature | Description |
|---------|-------------|
| **Log integration** | Ingests log streams from applications, services, and infrastructure components |
| **Fast, minimal indexing** | Only indexes metadata (labels), not log content — keeps storage costs low |
| **Precise filtering** | Uses LogQL to filter and query log streams by label and log content |
| **Container environment integrations** | Native support for Kubernetes, Docker, and cloud-native workloads |

Loki pairs with **Grafana Alloy** (formerly Promtail) for log shipping, and log panels in Grafana dashboards display Loki streams natively.

### Grafana Mimir

Mimir is Grafana's scalable, long-term storage backend for Prometheus metrics.

| Feature | Description |
|---------|-------------|
| **High availability** | Multi-replica ingestion and query path — no single point of failure |
| **Sharded query engine** | Parallelises large queries across multiple nodes for fast results |
| **Prometheus-compatible** | Drop-in replacement for Prometheus remote_write; accepts PromQL queries |

Mimir enables multi-tenancy so a single Mimir cluster can serve multiple teams or organisations.

### Grafana Tempo

Tempo is Grafana's distributed tracing backend.

| Feature | Description |
|---------|-------------|
| **Event tracing** | Captures distributed traces across microservices (spans, traces, trace IDs) |
| **Highly scalable** | Object-storage backend (S3/GCS/Azure) — scales to billions of spans |
| **Deep visibility** | Links traces to metrics and logs for full-stack root cause analysis |
| **Third-party integrations** | Compatible with OpenTelemetry, Jaeger, Zipkin, and other trace protocols |

**Trace → Metrics → Logs correlation:** Grafana's exemplar feature links a high-latency PromQL datapoint directly to the Tempo trace that caused it.

### Grafana Alloy

Alloy is the next-generation telemetry collector from Grafana Labs, replacing the legacy Promtail agent.

| Feature | Description |
|---------|-------------|
| **Replaces Promtail** | Drop-in replacement for log shipping to Loki; backwards-compatible |
| **Data transformations** | Built-in pipeline for filtering, relabelling, and enriching telemetry data |
| **Stack integrations** | Ships metrics to Prometheus/Mimir, logs to Loki, traces to Tempo in one agent |

Alloy uses a **River configuration language** (HCL-inspired) to define its telemetry pipeline.

### Grafana Product Ecosystem — At a Glance

| Product | Signal Type | Role |
|---------|-------------|------|
| **Grafana** | (all) | Visualisation and dashboarding platform |
| **Loki** | Logs | Log aggregation and querying (LogQL) |
| **Mimir** | Metrics | Scalable long-term Prometheus metrics storage |
| **Tempo** | Traces | Distributed trace storage and querying (TraceQL) |
| **Alloy** | Metrics/Logs/Traces | Unified telemetry collector (replaces Promtail) |
| **Prometheus** | Metrics | Time-series scraping and short-term storage |
| **OnCall** | Alerts | On-call scheduling and incident routing |
| **Pyroscope** | Profiles | Continuous profiling for performance analysis |

---
## Module 2 — Enabling Advanced Storage, Tracing, and Rules

### A Grafana Loki Stack — Architecture

A typical Loki deployment on a Linux server:

```
Server
├── Metrics  ──►  Prometheus
├── Logs     ──►  Prometheus
│                     │
│                     ▼
│                   Alloy  ──►  Loki  ──►  Grafana Dashboard
```

**Flow:**
1. Applications emit metrics and logs on the server
2. Prometheus scrapes metrics; Alloy tails log files
3. Alloy ships logs to Loki
4. Grafana queries Loki with LogQL and displays results in log panels

### Adding Mimir to Your Stack

To add Mimir as scalable metrics storage, configure Prometheus to remote_write to Mimir, then point Grafana at Mimir as its Prometheus data source.

Mimir requires a **blocks storage backend** — typically S3 or compatible object storage.

In [ ]:
# Mimir cloud storage configuration (mimir.yaml excerpt)
mimir_cloud_storage_config = """
blocks_storage:
  backend: s3
  s3:
    endpoint: s3.amazonaws.com
    bucket_name: my-mimir-rules
    region: us-east-1
    access_key_id: <AWS_ACCESS_KEY>
    secret_access_key: <AWS_SECRET_KEY>
"""
print(mimir_cloud_storage_config)

**Key configuration fields:**

| Field | Description |
|-------|-------------|
| `backend` | Storage type: `s3`, `gcs`, `azure`, or `filesystem` |
| `bucket_name` | S3 bucket that will hold the TSDB blocks |
| `region` | AWS region of the bucket |
| `access_key_id` | AWS credentials (prefer IAM roles in production) |

In [ ]:
# PromQL examples when querying a Mimir backend
mimir_queries = {
    "Total query requests by tenant and job": 
        "sum by (tenant, job) (mimir_global_tenant_query_requests_total)",
    "In-memory series per user (ingester)": 
        "sum by (user) (cortex_ingester_memory_series)",
}

for description, query in mimir_queries.items():
    print(f"{description}:")
    print(f"  {query}\n")

**Notes on Mimir-specific metrics:**

- `mimir_global_tenant_query_requests_total` — total queries received, broken down by tenant and job; useful for capacity planning and billing
- `cortex_ingester_memory_series` — number of active time series held in ingester memory per user; high values indicate heavy write loads

Mimir exposes Prometheus-style metrics about itself on port `8080` by default.

### Following Metric Traces With Tempo

Tempo enables **trace-to-metrics** and **metrics-to-trace** navigation:

1. **Exemplars** — Prometheus stores a trace ID alongside a metric sample; clicking the exemplar in a Grafana panel opens the matching Tempo trace
2. **Derived metrics** — Tempo generates RED metrics (Rate, Errors, Duration) from trace data via `spanmetrics` processor
3. **Service graph** — Tempo builds a dependency graph of services from trace relationships

**TraceQL** is Tempo's query language:
```traceql
{ .http.url = "/api/users" && duration > 200ms }
```
This finds all traces where the HTTP URL is `/api/users` and the span duration exceeded 200 ms.

---
## Module 3 — Advanced Grafana Integrations and Automation

### Building a Custom Dashboard

Advanced dashboard techniques beyond basic panel creation:

| Technique | How to Apply |
|-----------|-------------|
| **Template variables** | Create dashboard variables (`$instance`, `$job`) via Dashboard Settings → Variables; panels reference `$var` in queries |
| **Repeat rows/panels** | Set a panel's "Repeat by variable" to duplicate it for each variable value |
| **Data links** | Add clickable links inside panels that navigate to other dashboards or external URLs with interpolated variables |
| **Transformations** | Apply server-side transforms (merge, filter, calculate field) without changing queries |
| **Library panels** | Create reusable panels that can be shared across multiple dashboards |
| **Dashboard provisioning** | Deploy dashboards as JSON via `/etc/grafana/provisioning/dashboards/` for GitOps workflows |

### Applying Role-Based Access Controls (RBAC)

Grafana RBAC (available in Enterprise and Grafana Cloud) extends the four basic roles with fine-grained, provisioned custom roles.

**Basic (built-in) roles:**

| Role | Scope | Permissions |
|------|-------|-------------|
| **Grafana Admin** | Instance | Manages all organisations, users, and server settings |
| **Admin** | Organisation | Manages users, teams, data sources, and dashboards within an org |
| **Editor** | Organisation | Creates and edits dashboards; cannot manage users or data sources |
| **Viewer** | Organisation | Read-only access to dashboards and data |

In [ ]:
# Custom RBAC role provisioned via YAML
# File: /etc/grafana/provisioning/access-control/marketing_roles.yaml

rbac_yaml = """
roles:
  - name: MarketingDashboardEditor
    description: "Custom role for Marketing team."
    permissions:
      - action: "dashboards:edit"
        scope: "folders:uid:marketing-folder"
      - action: "dashboards:read"
        scope: "folders:uid:marketing-folder"
      - action: "datasources:query"
        scope: "datasources:uid:marketing-ds"
      - action: "explore:read"
        scope: "datasources:uid:marketing-ds"
      - action: "dashboards:delete"
        scope: "folders:uid:marketing-folder"
        allow: false
assignments:
  teams:
    - team: "Marketing Team"
      roles:
        - MarketingDashboardEditor
"""
print(rbac_yaml)

**RBAC YAML structure explained:**

| Field | Description |
|-------|-------------|
| `name` | Unique identifier for the custom role |
| `permissions[].action` | Permission string in `resource:action` format (e.g. `dashboards:edit`) |
| `permissions[].scope` | Resource UID to limit the permission (e.g. `folders:uid:marketing-folder`) |
| `permissions[].allow` | `false` explicitly denies the action — useful to override inherited permissions |
| `assignments.teams` | Assigns the custom role to one or more Grafana teams |

**Common RBAC action strings:**

| Action | Description |
|--------|-------------|
| `dashboards:read` | View dashboards |
| `dashboards:edit` | Edit existing dashboards |
| `dashboards:create` | Create new dashboards |
| `dashboards:delete` | Delete dashboards |
| `datasources:query` | Run queries against a data source |
| `explore:read` | Access the Explore view for ad hoc queries |
| `folders:read` | List and view dashboard folders |
| `users:read` | View user accounts (Admin-level) |

### Grafana HTTP API for Automation

Grafana exposes a REST API for automating common operations:

| Resource | Endpoint | Common Methods |
|----------|----------|-----------------|
| Dashboards | `/api/dashboards/db` | `POST` (create/update), `GET` (fetch by UID) |
| Folders | `/api/folders` | `GET`, `POST`, `DELETE` |
| Alert rules | `/api/v1/provisioning/alert-rules` | `GET`, `POST`, `PUT`, `DELETE` |
| Data sources | `/api/datasources` | `GET`, `POST`, `PUT`, `DELETE` |
| Users | `/api/admin/users` | `GET`, `POST`, `DELETE` |
| Teams | `/api/teams` | `GET`, `POST`, `PUT` |
| Service accounts | `/api/serviceaccounts` | `GET`, `POST`, `DELETE` |

**Authentication options:**
- **API key** (legacy) — static token passed as `Authorization: Bearer <token>`
- **Service account token** (recommended) — scoped token tied to a service account with a specific role

In [ ]:
import json

# Example: create a dashboard folder via the Grafana API
# POST /api/folders

create_folder_payload = {
    "uid": "marketing-folder",
    "title": "Marketing"
}

print("POST /api/folders")
print("Headers: { Authorization: Bearer <service_account_token> }")
print(json.dumps(create_folder_payload, indent=2))

### Service Accounts

Service accounts replace legacy API keys as the preferred method for non-human authentication:

- Created under **Administration → Service Accounts**
- Assigned an **org role** (Viewer / Editor / Admin)
- Issued one or more **tokens** with optional expiry dates
- Tokens are used in HTTP API calls: `Authorization: Bearer <token>`

Advantages over API keys:
- Visible in the audit log with a named identity
- Can be assigned fine-grained RBAC permissions
- Token rotation without changing org-level permissions

---
## Course Summary

| Module | Key Takeaways |
|--------|---------------|
| **Grafana Product Ecosystem** | Loki (logs), Mimir (metrics), Tempo (traces), Alloy (collector) form the LGTM stack alongside Grafana |
| **Advanced Storage, Tracing & Rules** | Mimir uses S3-backed block storage; Tempo links traces to metrics via exemplars; Loki+Alloy pipeline ships logs to dashboards |
| **Integrations & Automation** | RBAC provisions fine-grained permissions via YAML; Grafana HTTP API enables GitOps-style automation; service accounts replace API keys |

---
## Quick Reference

| Resource | URL / Path |
|----------|------------|
| Grafana Loki docs | https://grafana.com/docs/loki/latest/ |
| Grafana Mimir docs | https://grafana.com/docs/mimir/latest/ |
| Grafana Tempo docs | https://grafana.com/docs/tempo/latest/ |
| Grafana Alloy docs | https://grafana.com/docs/alloy/latest/ |
| RBAC provisioning | https://grafana.com/docs/grafana/latest/administration/roles-and-permissions/access-control/rbac-provisioning/ |
| Grafana HTTP API | https://grafana.com/docs/grafana/latest/developers/http_api/ |
| Service accounts | https://grafana.com/docs/grafana/latest/administration/service-accounts/ |
| Provisioning dir (access control) | `/etc/grafana/provisioning/access-control/` |
| Provisioning dir (dashboards) | `/etc/grafana/provisioning/dashboards/` |